# ⚡ Análisis de Series de Tiempo — HVAC AHU Blower 78
## Consumo Energético en KWh — Análisis Data-Driven Completo

---

## 📋 1. Contexto y Planteamiento del Problema

### 1.1 Contexto del Estudio

Los sistemas **HVAC (Heating, Ventilation and Air Conditioning)** son responsables de entre el 40% y el 60% del consumo energético en edificios comerciales e industriales. El componente **AHU Blower (Air Handling Unit)** es el motor eléctrico que impulsa el flujo de aire acondicionado a través de los ductos.

Este dataset fue recolectado mediante un **dispositivo IoT** que registra el consumo eléctrico del Blower 78 con intervalos de aproximadamente **10-15 minutos**, midiendo la energía consumida entre el timestamp actual y el anterior (en KWh).

### 1.2 Características del Sistema
- **Motor**: Capacidad fija en KW (potencia nominal constante)
- **Operación**: El consumo nulo (< 0.5 KWh) indica que la máquina estaba **apagada** durante ese slot
- **Naturaleza**: La serie es **estacionaria en media** por diseño (capacidad fija del motor)
- **Período**: Enero–Febrero 2022

### 1.3 Planteamiento del Problema

> **¿Cuáles son los patrones de consumo energético del Blower 78, existen anomalías operativas, y qué estructura temporal subyace a la serie?**

### 1.4 Objetivos del Análisis

| # | Objetivo | Tipo |
|---|----------|------|
| 1 | Explorar y limpiar el dataset unificado | Descriptivo |
| 2 | Analizar la distribución estadística del consumo | Descriptivo |
| 3 | Visualizar la serie temporal y sus componentes | Descriptivo |
| 4 | Detectar anomalías (picos, consumo fuera de rango) | Diagnóstico |
| 5 | Verificar estacionariedad (ADF, KPSS) | Analítico |
| 6 | Analizar autocorrelación y estructura temporal | Analítico |
| 7 | Descomponer la serie (tendencia, estacionalidad, residuo) | Analítico |
| 8 | Extraer insights accionables para gestión energética | Prescriptivo |

### 1.5 Insights Esperados
- ¿En qué horas/días el blower consume más energía?
- ¿Existen eventos anómalos (picos de consumo)?
- ¿Hay patrones diarios o semanales?
- ¿La serie es realmente estacionaria?
- ¿Cuánto tiempo opera el blower vs. tiempo apagado?

---
## 📦 2. Instalación e Importación de Librerías

In [ ]:
# Instalaciones necesarias para Colab
!pip install -q statsmodels plotly kaleido scipy missingno

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Stats
from scipy import stats
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Estilo visual global
plt.rcParams.update({
    'figure.facecolor': '#0f0f23',
    'axes.facecolor': '#1a1a2e',
    'axes.edgecolor': '#444466',
    'axes.labelcolor': '#e0e0ff',
    'xtick.color': '#aaaacc',
    'ytick.color': '#aaaacc',
    'text.color': '#e0e0ff',
    'grid.color': '#2a2a4a',
    'grid.linestyle': '--',
    'grid.alpha': 0.5,
    'font.family': 'DejaVu Sans',
    'font.size': 11,
})

# Paleta de colores principal
COLORS = {
    'primary':   '#7b61ff',
    'accent':    '#00d4ff',
    'warning':   '#ff6b6b',
    'success':   '#4ecdc4',
    'neutral':   '#f7dc6f',
    'bg_dark':   '#0f0f23',
    'bg_mid':    '#1a1a2e',
}

print('✅ Librerías cargadas correctamente')

---
## 📂 3. Carga y Unificación de Datos

> **Subir los 3 archivos CSV a Colab antes de ejecutar:**  
> `KwhConsumptionBlower78_1.csv`, `KwhConsumptionBlower78_2.csv`, `KwhConsumptionBlower78_3.csv`  
> Usar el panel lateral izquierdo → ícono de carpeta → subir archivos

In [ ]:
# ─── Carga de los 3 archivos ───────────────────────────────────────────────
files = [
    'KwhConsumptionBlower78_1.csv',
    'KwhConsumptionBlower78_2.csv',
    'KwhConsumptionBlower78_3.csv',
]

dfs = []
for f in files:
    df_temp = pd.read_csv(f)
    dfs.append(df_temp)
    print(f'📄 {f}: {len(df_temp):,} registros')

df_raw = pd.concat(dfs, ignore_index=True)
print(f'\n📊 Total combinado: {len(df_raw):,} registros')

# ─── Limpieza y parseo de fechas ──────────────────────────────────────────
df_raw['Datetime'] = pd.to_datetime(
    df_raw['TxnDate'].astype(str) + ' ' + df_raw['TxnTime'].astype(str),
    format='%d %b %Y %H:%M:%S'
)

df = df_raw[['Datetime', 'Consumption']].copy()
df = df.sort_values('Datetime').reset_index(drop=True)
df = df.drop_duplicates(subset='Datetime')

print(f'\n📅 Rango temporal: {df.Datetime.min()} → {df.Datetime.max()}')
print(f'🔢 Registros únicos: {len(df):,}')
df.head(10)

---
## 🔍 4. Análisis Exploratorio de Datos (EDA)

### 4.1 Estadísticas Descriptivas

In [ ]:
# ─── Estadísticas descriptivas ───────────────────────────────────────────
stats_df = df['Consumption'].describe().to_frame().T
stats_df['skewness'] = df['Consumption'].skew()
stats_df['kurtosis'] = df['Consumption'].kurtosis()
stats_df['cv_%'] = (df['Consumption'].std() / df['Consumption'].mean()) * 100
stats_df['zeros'] = (df['Consumption'] < 0.5).sum()
stats_df['zeros_%'] = stats_df['zeros'] / len(df) * 100
print('📊 Estadísticas Descriptivas del Consumo (KWh)')
print('='*60)
display(stats_df.round(4))

print(f"\n🔴 Registros con consumo < 0.5 KWh (máquina APAGADA): {(df['Consumption'] < 0.5).sum():,}")
print(f"🟢 Registros con consumo ≥ 0.5 KWh (máquina ENCENDIDA): {(df['Consumption'] >= 0.5).sum():,}")

In [ ]:
# ─── Gráfica 1: Dashboard de distribución estadística ────────────────────
fig = plt.figure(figsize=(18, 10), facecolor=COLORS['bg_dark'])
fig.suptitle('⚡ HVAC Blower 78 — Distribución del Consumo Energético (KWh)',
             fontsize=16, fontweight='bold', color='white', y=0.98)

gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# Panel 1: Histograma con KDE
ax1 = fig.add_subplot(gs[0, :])
ax1.set_facecolor(COLORS['bg_mid'])
consumption_all = df['Consumption'].values
ax1.hist(consumption_all, bins=80, color=COLORS['primary'], alpha=0.7,
         edgecolor='none', label='Todos los registros')
ax1.hist(consumption_all[consumption_all >= 0.5], bins=80,
         color=COLORS['accent'], alpha=0.6, edgecolor='none',
         label='Máquina ENCENDIDA (≥0.5 KWh)')
ax1.axvline(df['Consumption'].mean(), color=COLORS['warning'], linewidth=2,
            linestyle='--', label=f'Media: {df["Consumption"].mean():.3f} KWh')
ax1.axvline(df['Consumption'].median(), color=COLORS['neutral'], linewidth=2,
            linestyle=':', label=f'Mediana: {df["Consumption"].median():.3f} KWh')
ax1.axvline(0.5, color='#ff4444', linewidth=1.5, linestyle='-.',
            label='Umbral ON/OFF (0.5 KWh)')
ax1.set_xlabel('Consumo (KWh)', fontsize=12)
ax1.set_ylabel('Frecuencia', fontsize=12)
ax1.set_title('Histograma de Consumo', fontsize=13, color=COLORS['accent'])
ax1.legend(fontsize=10, facecolor='#1a1a2e', edgecolor='#7b61ff')
ax1.grid(True)

# Panel 2: Boxplot
ax2 = fig.add_subplot(gs[1, 0])
ax2.set_facecolor(COLORS['bg_mid'])
on_vals = df[df['Consumption'] >= 0.5]['Consumption']
off_vals = df[df['Consumption'] < 0.5]['Consumption']
bp = ax2.boxplot([on_vals, off_vals],
                 patch_artist=True, widths=0.5,
                 boxprops=dict(facecolor=COLORS['primary'], alpha=0.7),
                 medianprops=dict(color=COLORS['warning'], linewidth=2.5),
                 whiskerprops=dict(color=COLORS['accent']),
                 capprops=dict(color=COLORS['accent']),
                 flierprops=dict(marker='o', color=COLORS['warning'],
                                 markersize=3, alpha=0.5))
ax2.set_xticklabels(['ENCENDIDA\n(≥0.5 KWh)', 'APAGADA\n(<0.5 KWh)'])
ax2.set_ylabel('Consumo (KWh)')
ax2.set_title('Boxplot por Estado', fontsize=12, color=COLORS['accent'])
ax2.grid(True, axis='y')

# Panel 3: Violín
ax3 = fig.add_subplot(gs[1, 1])
ax3.set_facecolor(COLORS['bg_mid'])
violin = ax3.violinplot(on_vals.values, showmeans=True, showmedians=True)
for pc in violin['bodies']:
    pc.set_facecolor(COLORS['accent'])
    pc.set_alpha(0.7)
violin['cmeans'].set_color(COLORS['warning'])
violin['cmedians'].set_color(COLORS['neutral'])
ax3.set_ylabel('Consumo (KWh)')
ax3.set_xticks([1])
ax3.set_xticklabels(['Máquina ENCENDIDA'])
ax3.set_title('Violin Plot', fontsize=12, color=COLORS['accent'])
ax3.grid(True, axis='y')

# Panel 4: Pie ON vs OFF
ax4 = fig.add_subplot(gs[1, 2])
ax4.set_facecolor(COLORS['bg_mid'])
on_count = (df['Consumption'] >= 0.5).sum()
off_count = (df['Consumption'] < 0.5).sum()
wedges, texts, autotexts = ax4.pie(
    [on_count, off_count],
    labels=['ENCENDIDA', 'APAGADA'],
    colors=[COLORS['accent'], COLORS['warning']],
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops=dict(edgecolor='#0f0f23', linewidth=2)
)
for t in texts + autotexts:
    t.set_color('white')
ax4.set_title('Estado Operativo', fontsize=12, color=COLORS['accent'])

plt.savefig('01_distribucion.png', dpi=150, bbox_inches='tight',
            facecolor=COLORS['bg_dark'])
plt.show()
print('✅ Gráfica 1 generada')

---
## 📈 5. Visualización de la Serie Temporal Completa

In [ ]:
# ─── Gráfica 2: Serie temporal completa ──────────────────────────────────
fig = plt.figure(figsize=(20, 10), facecolor=COLORS['bg_dark'])
fig.suptitle('⚡ Serie Temporal Completa — Consumo KWh Blower 78\nEnero–Febrero 2022',
             fontsize=16, fontweight='bold', color='white')

gs = gridspec.GridSpec(2, 1, figure=fig, hspace=0.35, height_ratios=[3, 1])

ax1 = fig.add_subplot(gs[0])
ax1.set_facecolor(COLORS['bg_mid'])

# Área bajo la curva
ax1.fill_between(df['Datetime'], df['Consumption'],
                 color=COLORS['primary'], alpha=0.15)
ax1.plot(df['Datetime'], df['Consumption'],
         color=COLORS['primary'], linewidth=0.7, alpha=0.8, label='Consumo KWh')

# Resaltar anomalías (> 3 desviaciones estándar)
mean_c = df['Consumption'].mean()
std_c  = df['Consumption'].std()
anomalies = df[df['Consumption'] > mean_c + 3*std_c]
ax1.scatter(anomalies['Datetime'], anomalies['Consumption'],
            color=COLORS['warning'], s=40, zorder=5, label=f'Anomalías (>μ+3σ): {len(anomalies)}')

# Líneas de referencia
ax1.axhline(mean_c, color=COLORS['neutral'], linewidth=1.5,
            linestyle='--', label=f'Media: {mean_c:.3f} KWh')
ax1.axhline(mean_c + 3*std_c, color=COLORS['warning'], linewidth=1,
            linestyle=':', label=f'μ+3σ: {mean_c+3*std_c:.3f} KWh')
ax1.axhline(0.5, color='#ff4444', linewidth=1, linestyle='-.',
            label='Umbral ON/OFF')

ax1.set_ylabel('Consumo (KWh)', fontsize=12)
ax1.set_title('Serie Temporal Completa', fontsize=13, color=COLORS['accent'])
ax1.legend(fontsize=9, facecolor='#1a1a2e', edgecolor='#7b61ff', ncol=3)
ax1.grid(True)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))

# Panel inferior: media móvil
ax2 = fig.add_subplot(gs[1], sharex=ax1)
ax2.set_facecolor(COLORS['bg_mid'])
rolling_mean = df.set_index('Datetime')['Consumption'].rolling('24h').mean()
ax2.plot(rolling_mean.index, rolling_mean.values,
         color=COLORS['success'], linewidth=1.5, label='Media móvil 24h')
ax2.fill_between(rolling_mean.index, rolling_mean.values,
                 color=COLORS['success'], alpha=0.15)
ax2.set_ylabel('Media móvil\n(KWh)', fontsize=10)
ax2.set_xlabel('Fecha', fontsize=12)
ax2.legend(fontsize=9, facecolor='#1a1a2e', edgecolor='#7b61ff')
ax2.grid(True)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))

plt.savefig('02_serie_temporal.png', dpi=150, bbox_inches='tight',
            facecolor=COLORS['bg_dark'])
plt.show()
print('✅ Gráfica 2 generada')

---
## 🗓️ 6. Análisis de Patrones Temporales (Heatmap & Perfiles)

In [ ]:
# ─── Ingeniería de features temporales ───────────────────────────────────
df['hour']        = df['Datetime'].dt.hour
df['day']         = df['Datetime'].dt.day
df['month']       = df['Datetime'].dt.month
df['dayofweek']   = df['Datetime'].dt.dayofweek
df['week']        = df['Datetime'].dt.isocalendar().week.astype(int)
df['day_name']    = df['Datetime'].dt.day_name()
df['is_on']       = (df['Consumption'] >= 0.5).astype(int)
df['date_only']   = df['Datetime'].dt.date

# ─── Gráfica 3: Heatmap hora x día ────────────────────────────────────────
pivot = df.pivot_table(values='Consumption', index='hour',
                        columns='date_only', aggfunc='mean')

fig, axes = plt.subplots(1, 2, figsize=(20, 8), facecolor=COLORS['bg_dark'])
fig.suptitle('🗓️ Patrones Temporales — Consumo por Hora y Día',
             fontsize=16, fontweight='bold', color='white')

# Heatmap principal
ax = axes[0]
ax.set_facecolor(COLORS['bg_mid'])
sns.heatmap(pivot, ax=ax,
            cmap='viridis', linewidths=0.0,
            cbar_kws={'label': 'Consumo promedio (KWh)',
                       'shrink': 0.8},
            yticklabels=True)
ax.set_title('Consumo Promedio: Hora del Día × Fecha',
             fontsize=13, color=COLORS['accent'])
ax.set_xlabel('Fecha', fontsize=12)
ax.set_ylabel('Hora del Día', fontsize=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=7)

# Perfil horario promedio
ax2 = axes[1]
ax2.set_facecolor(COLORS['bg_mid'])
hourly_profile = df.groupby('hour')['Consumption'].agg(['mean', 'std'])
hours = hourly_profile.index
ax2.fill_between(hours,
                 hourly_profile['mean'] - hourly_profile['std'],
                 hourly_profile['mean'] + hourly_profile['std'],
                 color=COLORS['primary'], alpha=0.3, label='±1 Desv. Estándar')
ax2.plot(hours, hourly_profile['mean'],
         color=COLORS['accent'], linewidth=2.5, marker='o',
         markersize=5, label='Consumo promedio')
ax2.set_xlabel('Hora del Día', fontsize=12)
ax2.set_ylabel('Consumo promedio (KWh)', fontsize=12)
ax2.set_title('Perfil Horario Promedio (±1σ)', fontsize=13, color=COLORS['accent'])
ax2.set_xticks(range(0, 24, 2))
ax2.legend(fontsize=10, facecolor='#1a1a2e', edgecolor='#7b61ff')
ax2.grid(True)

# Sombrear horas nocturnas
ax2.axvspan(0, 6, color='#7b61ff', alpha=0.08, label='Noche')
ax2.axvspan(22, 24, color='#7b61ff', alpha=0.08)

plt.savefig('03_patrones_temporales.png', dpi=150, bbox_inches='tight',
            facecolor=COLORS['bg_dark'])
plt.show()
print('✅ Gráfica 3 generada')

In [ ]:
# ─── Gráfica 4: Perfil semanal y diario ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7), facecolor=COLORS['bg_dark'])
fig.suptitle('📅 Consumo por Día de la Semana y Mes',
             fontsize=16, fontweight='bold', color='white')

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday',
             'Friday', 'Saturday', 'Sunday']
day_labels = ['Lun', 'Mar', 'Mié', 'Jue', 'Vie', 'Sáb', 'Dom']

weekly = df.groupby('day_name')['Consumption'].agg(['mean', 'std']).reindex(day_order)

ax1 = axes[0]
ax1.set_facecolor(COLORS['bg_mid'])
colors_bar = [COLORS['primary'] if i < 5 else COLORS['warning']
              for i in range(7)]
bars = ax1.bar(range(7), weekly['mean'], color=colors_bar,
               alpha=0.85, edgecolor='none', width=0.6)
ax1.errorbar(range(7), weekly['mean'], yerr=weekly['std'],
             fmt='none', color=COLORS['accent'], capsize=5, linewidth=2)
ax1.set_xticks(range(7))
ax1.set_xticklabels(day_labels, fontsize=11)
ax1.set_ylabel('Consumo promedio (KWh)', fontsize=12)
ax1.set_title('Consumo por Día de la Semana', fontsize=13, color=COLORS['accent'])
ax1.grid(True, axis='y')
for bar, val in zip(bars, weekly['mean']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f'{val:.2f}', ha='center', va='bottom', fontsize=9,
             color='white', fontweight='bold')

# Consumo por mes
ax2 = axes[1]
ax2.set_facecolor(COLORS['bg_mid'])
monthly = df.groupby('month')['Consumption'].agg(['mean', 'std', 'count'])
month_names = {1: 'Enero 2022', 2: 'Febrero 2022'}
bars2 = ax2.bar([month_names.get(m, m) for m in monthly.index],
                monthly['mean'],
                color=[COLORS['accent'], COLORS['success']],
                alpha=0.85, edgecolor='none', width=0.4)
ax2.errorbar(range(len(monthly)), monthly['mean'], yerr=monthly['std'],
             fmt='none', color=COLORS['warning'], capsize=8, linewidth=2.5)
ax2.set_ylabel('Consumo promedio (KWh)', fontsize=12)
ax2.set_title('Consumo Promedio por Mes', fontsize=13, color=COLORS['accent'])
ax2.grid(True, axis='y')
for bar, val, cnt in zip(bars2, monthly['mean'], monthly['count']):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f'{val:.3f} KWh\n({cnt} muestras)', ha='center', va='bottom',
             fontsize=10, color='white', fontweight='bold')

plt.savefig('04_perfil_semanal_mensual.png', dpi=150, bbox_inches='tight',
            facecolor=COLORS['bg_dark'])
plt.show()
print('✅ Gráfica 4 generada')

---
## 🚨 7. Detección de Anomalías

In [ ]:
# ─── Detección de anomalías: Z-score + IQR ────────────────────────────────
df_on = df[df['Consumption'] >= 0.5].copy()

# Z-score
df_on['z_score'] = np.abs(stats.zscore(df_on['Consumption']))
df_on['anomaly_z'] = df_on['z_score'] > 3

# IQR
Q1, Q3 = df_on['Consumption'].quantile([0.25, 0.75])
IQR = Q3 - Q1
lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
df_on['anomaly_iqr'] = (df_on['Consumption'] < lower) | (df_on['Consumption'] > upper)

df_on['anomaly_any'] = df_on['anomaly_z'] | df_on['anomaly_iqr']

print(f'📊 Anomalías detectadas (Z-score >3): {df_on["anomaly_z"].sum()}')
print(f'📊 Anomalías detectadas (IQR): {df_on["anomaly_iqr"].sum()}')
print(f'📊 Anomalías totales (unión): {df_on["anomaly_any"].sum()}')
print(f'\nTop 10 consumos más altos:')
display(df_on.nlargest(10, 'Consumption')[['Datetime', 'Consumption', 'z_score']].round(3))

# ─── Gráfica 5: Anomaly Detection ─────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(20, 12), facecolor=COLORS['bg_dark'])
fig.suptitle('🚨 Detección de Anomalías — Consumo Blower 78',
             fontsize=16, fontweight='bold', color='white')

# Panel superior: serie con anomalías
ax1 = axes[0]
ax1.set_facecolor(COLORS['bg_mid'])
normal = df_on[~df_on['anomaly_any']]
anom   = df_on[df_on['anomaly_any']]

ax1.scatter(normal['Datetime'], normal['Consumption'],
            color=COLORS['accent'], s=8, alpha=0.5, label='Normal')
ax1.scatter(anom['Datetime'], anom['Consumption'],
            color=COLORS['warning'], s=60, zorder=5, marker='X',
            label=f'Anomalía ({len(anom)} puntos)', edgecolors='white', linewidths=0.5)
ax1.axhline(upper, color=COLORS['warning'], linewidth=1.5, linestyle='--',
            label=f'Límite superior IQR: {upper:.3f}')
ax1.axhline(lower, color=COLORS['neutral'], linewidth=1.5, linestyle='--',
            label=f'Límite inferior IQR: {lower:.3f}')
ax1.set_ylabel('Consumo (KWh)', fontsize=12)
ax1.set_title('Serie Temporal con Anomalías Marcadas', fontsize=13, color=COLORS['accent'])
ax1.legend(fontsize=10, facecolor='#1a1a2e', edgecolor='#7b61ff', ncol=2)
ax1.grid(True)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))

# Panel inferior: Z-score a lo largo del tiempo
ax2 = axes[1]
ax2.set_facecolor(COLORS['bg_mid'])
ax2.plot(df_on['Datetime'], df_on['z_score'],
         color=COLORS['primary'], linewidth=0.7, alpha=0.8)
ax2.fill_between(df_on['Datetime'], df_on['z_score'],
                 color=COLORS['primary'], alpha=0.2)
ax2.axhline(3, color=COLORS['warning'], linewidth=2, linestyle='--',
            label='Umbral Z=3')
ax2.scatter(df_on[df_on['anomaly_z']]['Datetime'],
            df_on[df_on['anomaly_z']]['z_score'],
            color=COLORS['warning'], s=50, zorder=5, marker='X')
ax2.set_ylabel('Z-Score', fontsize=12)
ax2.set_xlabel('Fecha', fontsize=12)
ax2.set_title('Z-Score del Consumo a lo Largo del Tiempo', fontsize=13, color=COLORS['accent'])
ax2.legend(fontsize=10, facecolor='#1a1a2e', edgecolor='#7b61ff')
ax2.grid(True)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))

plt.savefig('05_anomalias.png', dpi=150, bbox_inches='tight',
            facecolor=COLORS['bg_dark'])
plt.show()
print('✅ Gráfica 5 generada')

---
## 📉 8. Pruebas de Estacionariedad (ADF & KPSS)

In [ ]:
# ─── Trabajar solo con máquina encendida, serie indexada por tiempo ───────
ts = df_on.set_index('Datetime')['Consumption'].sort_index()
ts_resampled = ts.resample('15min').mean().dropna()

# ─── Prueba ADF (Augmented Dickey-Fuller) ────────────────────────────────
adf_result = adfuller(ts_resampled, autolag='AIC')
print('=== Augmented Dickey-Fuller Test ===')
print(f'ADF Statistic : {adf_result[0]:.6f}')
print(f'p-value       : {adf_result[1]:.6f}')
print(f'Lags used     : {adf_result[2]}')
print(f'Obs. used     : {adf_result[3]}')
print('Critical values:')
for key, val in adf_result[4].items():
    print(f'  {key}: {val:.6f}')
print(f"\n→ ¿Serie ESTACIONARIA? {'✅ SÍ' if adf_result[1] < 0.05 else '❌ NO'} (p={'%.4f' % adf_result[1]})")

print('\n=== KPSS Test ===')
kpss_result = kpss(ts_resampled, regression='c', nlags='auto')
print(f'KPSS Statistic: {kpss_result[0]:.6f}')
print(f'p-value       : {kpss_result[1]:.6f}')
print('Critical values:')
for key, val in kpss_result[3].items():
    print(f'  {key}: {val:.6f}')
print(f"\n→ ¿Serie ESTACIONARIA? {'✅ SÍ' if kpss_result[1] > 0.05 else '❌ NO'} (p={'%.4f' % kpss_result[1]})")

In [ ]:
# ─── Gráfica 6: Diagnóstico de estacionariedad ────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 10), facecolor=COLORS['bg_dark'])
fig.suptitle('📉 Análisis de Estacionariedad — Blower 78',
             fontsize=16, fontweight='bold', color='white')

# Media y varianza rolling
roll_mean = ts_resampled.rolling(window=48).mean()   # ~12h
roll_std  = ts_resampled.rolling(window=48).std()

ax1 = axes[0, 0]
ax1.set_facecolor(COLORS['bg_mid'])
ax1.plot(ts_resampled.index, ts_resampled.values,
         color=COLORS['primary'], alpha=0.6, linewidth=0.7, label='Serie original')
ax1.plot(roll_mean.index, roll_mean.values,
         color=COLORS['accent'], linewidth=2, label='Media móvil (48 obs)')
ax1.set_title('Serie + Media Móvil', fontsize=12, color=COLORS['accent'])
ax1.legend(fontsize=9, facecolor='#1a1a2e', edgecolor='#7b61ff')
ax1.grid(True)
ax1.set_ylabel('KWh')

ax2 = axes[0, 1]
ax2.set_facecolor(COLORS['bg_mid'])
ax2.plot(roll_std.index, roll_std.values,
         color=COLORS['warning'], linewidth=1.5, label='Desv. Estándar móvil')
ax2.fill_between(roll_std.index, roll_std.values,
                 color=COLORS['warning'], alpha=0.2)
ax2.set_title('Desviación Estándar Móvil', fontsize=12, color=COLORS['accent'])
ax2.legend(fontsize=9, facecolor='#1a1a2e', edgecolor='#7b61ff')
ax2.grid(True)
ax2.set_ylabel('KWh')

# Q-Q plot
ax3 = axes[1, 0]
ax3.set_facecolor(COLORS['bg_mid'])
qq = stats.probplot(ts_resampled.values, dist='norm')
ax3.scatter(qq[0][0], qq[0][1], color=COLORS['accent'],
            s=10, alpha=0.6, label='Datos observados')
xmin, xmax = qq[0][0].min(), qq[0][0].max()
yfit = qq[1][1] + qq[1][0] * np.array([xmin, xmax])
ax3.plot([xmin, xmax], yfit, color=COLORS['warning'], linewidth=2,
         label='Distribución normal teórica')
ax3.set_title('Q-Q Plot (Normalidad)', fontsize=12, color=COLORS['accent'])
ax3.set_xlabel('Cuantiles Teóricos')
ax3.set_ylabel('Cuantiles Muestral')
ax3.legend(fontsize=9, facecolor='#1a1a2e', edgecolor='#7b61ff')
ax3.grid(True)

# Resumen pruebas
ax4 = axes[1, 1]
ax4.set_facecolor(COLORS['bg_mid'])
ax4.axis('off')
adf_sig = '✅ ESTACIONARIA' if adf_result[1] < 0.05 else '❌ NO ESTACIONARIA'
kpss_sig = '✅ ESTACIONARIA' if kpss_result[1] > 0.05 else '❌ NO ESTACIONARIA'
table_data = [
    ['Prueba', 'Estadístico', 'p-valor', 'Resultado'],
    ['ADF', f'{adf_result[0]:.4f}', f'{adf_result[1]:.4f}', adf_sig],
    ['KPSS', f'{kpss_result[0]:.4f}', f'{kpss_result[1]:.4f}', kpss_sig],
]
table = ax4.table(cellText=table_data[1:], colLabels=table_data[0],
                  loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 2.5)
for (r, c), cell in table.get_celld().items():
    cell.set_facecolor('#1a1a2e' if r > 0 else '#7b61ff')
    cell.set_text_props(color='white')
    cell.set_edgecolor('#444466')
ax4.set_title('Resumen Pruebas de Estacionariedad',
              fontsize=12, color=COLORS['accent'])

plt.savefig('06_estacionariedad.png', dpi=150, bbox_inches='tight',
            facecolor=COLORS['bg_dark'])
plt.show()
print('✅ Gráfica 6 generada')

---
## 🔄 9. Autocorrelación (ACF & PACF)

In [ ]:
# ─── Gráfica 7: ACF y PACF ────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(18, 10), facecolor=COLORS['bg_dark'])
fig.suptitle('🔄 Autocorrelación — ACF & PACF del Consumo Blower 78',
             fontsize=16, fontweight='bold', color='white')

# ACF
acf_vals, confint_acf = acf(ts_resampled.values, nlags=96, alpha=0.05)
lags = np.arange(len(acf_vals))

ax1 = axes[0]
ax1.set_facecolor(COLORS['bg_mid'])
ax1.bar(lags, acf_vals, color=COLORS['accent'], alpha=0.7, width=0.5)
ax1.fill_between(lags, confint_acf[:, 0] - acf_vals,
                 confint_acf[:, 1] - acf_vals,
                 color=COLORS['warning'], alpha=0.25, label='Intervalo confianza 95%')
ax1.axhline(0, color='white', linewidth=1)
# Marcar lags múltiplos de 24h (96 obs @ 15min)
for lag in [96]:
    if lag < len(acf_vals):
        ax1.axvline(lag, color=COLORS['success'], linewidth=1.5,
                    linestyle='--', alpha=0.8, label=f'Lag 24h ({lag} obs)')
ax1.set_title('ACF — Función de Autocorrelación', fontsize=13, color=COLORS['accent'])
ax1.set_xlabel('Lag (intervalos de 15 min)', fontsize=11)
ax1.set_ylabel('Autocorrelación', fontsize=11)
ax1.legend(fontsize=10, facecolor='#1a1a2e', edgecolor='#7b61ff')
ax1.grid(True)

# PACF
pacf_vals, confint_pacf = pacf(ts_resampled.values, nlags=48, alpha=0.05,
                                method='ywm')
lags_p = np.arange(len(pacf_vals))

ax2 = axes[1]
ax2.set_facecolor(COLORS['bg_mid'])
ax2.bar(lags_p, pacf_vals, color=COLORS['primary'], alpha=0.7, width=0.5)
ax2.fill_between(lags_p, confint_pacf[:, 0] - pacf_vals,
                 confint_pacf[:, 1] - pacf_vals,
                 color=COLORS['warning'], alpha=0.25, label='Intervalo confianza 95%')
ax2.axhline(0, color='white', linewidth=1)
ax2.set_title('PACF — Función de Autocorrelación Parcial', fontsize=13, color=COLORS['accent'])
ax2.set_xlabel('Lag (intervalos de 15 min)', fontsize=11)
ax2.set_ylabel('Autocorrelación Parcial', fontsize=11)
ax2.legend(fontsize=10, facecolor='#1a1a2e', edgecolor='#7b61ff')
ax2.grid(True)

plt.savefig('07_acf_pacf.png', dpi=150, bbox_inches='tight',
            facecolor=COLORS['bg_dark'])
plt.show()
print('✅ Gráfica 7 generada')

---
## 🧩 10. Descomposición de la Serie Temporal

In [ ]:
# ─── Descomposición STL-like con statsmodels ──────────────────────────────
# Usamos periodo = 96 (96 intervalos de 15min = 24 horas)
decomp = seasonal_decompose(ts_resampled, model='additive', period=96)

# ─── Gráfica 8: Componentes de la serie ───────────────────────────────────
fig, axes = plt.subplots(4, 1, figsize=(20, 16), facecolor=COLORS['bg_dark'])
fig.suptitle('🧩 Descomposición de la Serie Temporal (Modelo Aditivo)\nPeríodo = 96 obs (24 horas)',
             fontsize=16, fontweight='bold', color='white')

components = [
    ('Serie Original', ts_resampled, COLORS['primary']),
    ('Tendencia', decomp.trend, COLORS['accent']),
    ('Estacionalidad', decomp.seasonal, COLORS['success']),
    ('Residuo', decomp.resid, COLORS['warning']),
]

for ax, (title, data, color) in zip(axes, components):
    ax.set_facecolor(COLORS['bg_mid'])
    if title == 'Residuo':
        ax.bar(data.index, data.values, color=color, alpha=0.5, width=0.008)
    else:
        ax.plot(data.index, data.values, color=color, linewidth=1.2)
        ax.fill_between(data.index, data.values, alpha=0.1, color=color)
    ax.axhline(0, color='white', linewidth=0.5, alpha=0.5)
    ax.set_ylabel('KWh', fontsize=10)
    ax.set_title(title, fontsize=12, color=color)
    ax.grid(True)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))

axes[-1].set_xlabel('Fecha', fontsize=12)

plt.savefig('08_descomposicion.png', dpi=150, bbox_inches='tight',
            facecolor=COLORS['bg_dark'])
plt.show()
print('✅ Gráfica 8 generada')

---
## ⚙️ 11. Análisis Operativo: Tiempo ON vs OFF y Consumo Acumulado

In [ ]:
# ─── Gráfica 9: Operación y consumo acumulado ─────────────────────────────
df_daily = df.groupby('date_only').agg(
    total_kwh=('Consumption', 'sum'),
    on_slots=('is_on', 'sum'),
    total_slots=('is_on', 'count'),
    max_kwh=('Consumption', 'max'),
    mean_kwh=('Consumption', 'mean')
).reset_index()
df_daily['on_pct'] = df_daily['on_slots'] / df_daily['total_slots'] * 100
df_daily['cumulative_kwh'] = df_daily['total_kwh'].cumsum()
df_daily['date_dt'] = pd.to_datetime(df_daily['date_only'])

fig, axes = plt.subplots(2, 2, figsize=(20, 12), facecolor=COLORS['bg_dark'])
fig.suptitle('⚙️ Análisis Operativo Diario — HVAC Blower 78',
             fontsize=16, fontweight='bold', color='white')

# Consumo total diario
ax1 = axes[0, 0]
ax1.set_facecolor(COLORS['bg_mid'])
ax1.bar(df_daily['date_dt'], df_daily['total_kwh'],
        color=COLORS['primary'], alpha=0.8, width=0.7)
ax1.plot(df_daily['date_dt'], df_daily['total_kwh'].rolling(7).mean(),
         color=COLORS['warning'], linewidth=2, label='Media móvil 7d')
ax1.set_title('Consumo Total Diario (KWh)', fontsize=12, color=COLORS['accent'])
ax1.set_ylabel('KWh')
ax1.legend(fontsize=9, facecolor='#1a1a2e', edgecolor='#7b61ff')
ax1.grid(True, axis='y')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')

# % operación diaria
ax2 = axes[0, 1]
ax2.set_facecolor(COLORS['bg_mid'])
sc = ax2.scatter(df_daily['date_dt'], df_daily['on_pct'],
                 c=df_daily['on_pct'], cmap='RdYlGn', s=60,
                 vmin=0, vmax=100, zorder=3)
ax2.fill_between(df_daily['date_dt'], df_daily['on_pct'],
                 color=COLORS['success'], alpha=0.15)
ax2.plot(df_daily['date_dt'], df_daily['on_pct'],
         color=COLORS['success'], linewidth=1.5, alpha=0.7)
plt.colorbar(sc, ax=ax2, label='% ON')
ax2.set_title('% de Tiempo Operativo por Día', fontsize=12, color=COLORS['accent'])
ax2.set_ylabel('% Tiempo ON')
ax2.set_ylim(0, 105)
ax2.grid(True)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Consumo acumulado
ax3 = axes[1, 0]
ax3.set_facecolor(COLORS['bg_mid'])
ax3.fill_between(df_daily['date_dt'], df_daily['cumulative_kwh'],
                 color=COLORS['accent'], alpha=0.25)
ax3.plot(df_daily['date_dt'], df_daily['cumulative_kwh'],
         color=COLORS['accent'], linewidth=2.5)
ax3.set_title('Consumo Energético Acumulado (KWh)', fontsize=12, color=COLORS['accent'])
ax3.set_ylabel('KWh Acumulado')
ax3.set_xlabel('Fecha')
ax3.grid(True)
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
plt.setp(ax3.xaxis.get_majorticklabels(), rotation=45, ha='right')
total_kwh_val = df_daily['cumulative_kwh'].iloc[-1]
ax3.annotate(f'Total: {total_kwh_val:.1f} KWh',
             xy=(df_daily['date_dt'].iloc[-1], total_kwh_val),
             xytext=(-60, -30), textcoords='offset points',
             color='white', fontweight='bold', fontsize=11,
             arrowprops=dict(arrowstyle='->', color=COLORS['accent']))

# Relación consumo max vs promedio
ax4 = axes[1, 1]
ax4.set_facecolor(COLORS['bg_mid'])
sc2 = ax4.scatter(df_daily['mean_kwh'], df_daily['max_kwh'],
                  c=df_daily['on_pct'], cmap='plasma',
                  s=50, alpha=0.8)
plt.colorbar(sc2, ax=ax4, label='% ON')
ax4.set_xlabel('Consumo Promedio Diario (KWh)', fontsize=11)
ax4.set_ylabel('Pico de Consumo Diario (KWh)', fontsize=11)
ax4.set_title('Promedio vs Pico de Consumo', fontsize=12, color=COLORS['accent'])
ax4.grid(True)
# Línea de referencia 1:1
min_v = min(df_daily['mean_kwh'].min(), df_daily['max_kwh'].min())
max_v = max(df_daily['mean_kwh'].max(), df_daily['max_kwh'].max())
ax4.plot([min_v, max_v], [min_v, max_v], '--', color='gray', alpha=0.4, label='y=x')

plt.savefig('09_operativo.png', dpi=150, bbox_inches='tight',
            facecolor=COLORS['bg_dark'])
plt.show()
print('✅ Gráfica 9 generada')

---
## 🌐 12. Visualización Interactiva con Plotly

In [ ]:
# ─── Gráfica 10: Dashboard interactivo Plotly ─────────────────────────────
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=(
        'Serie Temporal + Media Móvil 24h',
        'Distribución KDE por Estado',
        'Consumo Total Diario',
        'Heatmap: Hora × Día de Semana',
        '% Operación Diaria',
        'Consumo Acumulado',
    ),
    specs=[
        [{'colspan': 2}, None],
        [{'type': 'xy'}, {'type': 'xy'}],
        [{'type': 'xy'}, {'type': 'xy'}],
    ],
    vertical_spacing=0.1,
    horizontal_spacing=0.08,
)

# Fila 1: Serie temporal
fig.add_trace(go.Scatter(
    x=df['Datetime'], y=df['Consumption'],
    mode='lines', name='Consumo KWh',
    line=dict(color='#7b61ff', width=0.8),
    opacity=0.7
), row=1, col=1)
rolling_24h = df.set_index('Datetime')['Consumption'].rolling('24h').mean()
fig.add_trace(go.Scatter(
    x=rolling_24h.index, y=rolling_24h.values,
    mode='lines', name='Media móvil 24h',
    line=dict(color='#00d4ff', width=2)
), row=1, col=1)

# Fila 2 izq: Consumo diario
fig.add_trace(go.Bar(
    x=df_daily['date_dt'], y=df_daily['total_kwh'],
    name='Consumo diario', marker_color='#4ecdc4', opacity=0.8,
    showlegend=False
), row=2, col=1)

# Fila 2 der: Heatmap hora x día de semana
pivot_hw = df.pivot_table(values='Consumption',
                           index='hour', columns='dayofweek', aggfunc='mean')
fig.add_trace(go.Heatmap(
    z=pivot_hw.values,
    x=['Lun', 'Mar', 'Mié', 'Jue', 'Vie', 'Sáb', 'Dom'],
    y=list(range(24)),
    colorscale='Viridis',
    name='Heatmap',
    showlegend=False
), row=2, col=2)

# Fila 3 izq: % operación
fig.add_trace(go.Scatter(
    x=df_daily['date_dt'], y=df_daily['on_pct'],
    mode='lines+markers', name='% ON',
    line=dict(color='#f7dc6f', width=1.5),
    fill='tozeroy', fillcolor='rgba(247,220,111,0.1)',
    showlegend=False
), row=3, col=1)

# Fila 3 der: acumulado
fig.add_trace(go.Scatter(
    x=df_daily['date_dt'], y=df_daily['cumulative_kwh'],
    mode='lines', name='KWh Acumulado',
    line=dict(color='#ff6b6b', width=2),
    fill='tozeroy', fillcolor='rgba(255,107,107,0.1)',
    showlegend=False
), row=3, col=2)

fig.update_layout(
    height=900,
    template='plotly_dark',
    title_text='⚡ Dashboard Interactivo — HVAC Blower 78',
    title_x=0.5,
    title_font_size=18,
    paper_bgcolor='#0f0f23',
    plot_bgcolor='#1a1a2e',
    font=dict(color='#e0e0ff'),
    legend=dict(bgcolor='#1a1a2e', bordercolor='#7b61ff')
)

fig.show()
print('✅ Dashboard interactivo generado')

---
## 💡 13. Resumen de Insights y Conclusiones

In [ ]:
# ─── Cálculo de insights finales ──────────────────────────────────────────
total_kwh      = df['Consumption'].sum()
avg_daily      = df_daily['total_kwh'].mean()
on_pct_global  = (df['Consumption'] >= 0.5).mean() * 100
peak_hour      = df_on.groupby('hour')['Consumption'].mean().idxmax()
low_hour       = df_on.groupby('hour')['Consumption'].mean().idxmin()
anom_pct       = len(anom) / len(df_on) * 100
peak_day       = df_daily.loc[df_daily['total_kwh'].idxmax(), 'date_only']

print('\n' + '='*60)
print('       💡 INSIGHTS FINALES — HVAC BLOWER 78')
print('='*60)

insights = [
    ('⚡ Consumo Total del Período',
     f'{total_kwh:.1f} KWh (Enero–Febrero 2022)'),
    ('📅 Consumo Promedio Diario',
     f'{avg_daily:.2f} KWh/día'),
    ('🟢 Tasa de Operación Global',
     f'{on_pct_global:.1f}% del tiempo el blower estuvo encendido'),
    ('📈 Hora de Mayor Consumo Promedio',
     f'{peak_hour}:00 h → mayor demanda operacional'),
    ('📉 Hora de Menor Consumo Promedio',
     f'{low_hour}:00 h → posible ventana de mantenimiento'),
    ('🚨 Anomalías Detectadas',
     f'{len(anom)} registros ({anom_pct:.2f}%) superan μ+3σ'),
    ('📊 Estacionariedad ADF',
     f'p={adf_result[1]:.4f} → {"Serie ESTACIONARIA ✅" if adf_result[1]<0.05 else "No estacionaria ❌"}'),
    ('📊 Estacionariedad KPSS',
     f'p={kpss_result[1]:.4f} → {"Serie ESTACIONARIA ✅" if kpss_result[1]>0.05 else "No estacionaria ❌"}'),
    ('🔺 Pico de Consumo Máximo',
     f'{df["Consumption"].max():.3f} KWh (posible arranque o falla)'),
    ('📆 Día con Mayor Consumo Total',
     f'{peak_day}'),
]

for title, body in insights:
    print(f'\n{title}')
    print(f'   → {body}')

print('\n' + '='*60)
print('\n📌 RECOMENDACIONES OPERATIVAS:')
print(f'  1. Programar mantenimiento preventivo en hora {low_hour}:00 h (menor carga)')
print(f'  2. Investigar los {len(anom)} picos anómalos para descartar fallos mecánicos')
print(f'  3. Optimizar el scheduling de operación en las horas de menor demanda')
print(f'  4. La serie es estacionaria → viable usar modelos ARIMA/SARIMA para forecasting')
print(f'  5. El patrón horario es repetible → aplicable control predictivo (MPC)')

In [ ]:
# ─── Gráfica 11: Resumen visual final ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 7), facecolor=COLORS['bg_dark'])
fig.suptitle('💡 Resumen Ejecutivo — Insights del Análisis',
             fontsize=16, fontweight='bold', color='white')

# Comparativa mes a mes
ax1 = axes[0]
ax1.set_facecolor(COLORS['bg_mid'])
month_data = df.groupby('month').agg(
    total=('Consumption', 'sum'),
    mean=('Consumption', 'mean'),
    on_rate=('is_on', 'mean')
)
x = np.arange(2)
bars_t = ax1.bar(x - 0.2, month_data['total'] / 1000, 0.35,
                 color=COLORS['primary'], alpha=0.85, label='Consumo total (MWh)')
ax1b = ax1.twinx()
ax1b.set_facecolor(COLORS['bg_mid'])
bars_r = ax1b.bar(x + 0.2, month_data['on_rate'] * 100, 0.35,
                  color=COLORS['success'], alpha=0.85, label='% ON')
ax1.set_xticks(x)
ax1.set_xticklabels(['Enero\n2022', 'Febrero\n2022'])
ax1.set_ylabel('Consumo (MWh)', color=COLORS['primary'])
ax1b.set_ylabel('% Tiempo ON', color=COLORS['success'])
ax1.set_title('Comparativa Mensual', fontsize=12, color=COLORS['accent'])
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1b.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=8,
           facecolor='#1a1a2e', edgecolor='#7b61ff')
ax1.grid(True, axis='y', alpha=0.4)

# Ranking de horas
ax2 = axes[1]
ax2.set_facecolor(COLORS['bg_mid'])
hourly_rank = df_on.groupby('hour')['Consumption'].mean().sort_values(ascending=True)
colors_h = [COLORS['warning'] if h == peak_hour else
            COLORS['success'] if h == low_hour else
            COLORS['primary'] for h in hourly_rank.index]
ax2.barh(hourly_rank.index, hourly_rank.values, color=colors_h, alpha=0.85)
ax2.set_xlabel('Consumo promedio (KWh)', fontsize=11)
ax2.set_ylabel('Hora del Día', fontsize=11)
ax2.set_title('Ranking de Consumo por Hora', fontsize=12, color=COLORS['accent'])
ax2.axvline(hourly_rank.mean(), color=COLORS['neutral'], linestyle='--',
            linewidth=1.5, label='Media global')
ax2.legend(fontsize=9, facecolor='#1a1a2e', edgecolor='#7b61ff')
ax2.grid(True, axis='x', alpha=0.4)

# Speedometer / KPI text
ax3 = axes[2]
ax3.set_facecolor(COLORS['bg_mid'])
ax3.axis('off')
kpis = [
    ('⚡ Consumo Total', f'{total_kwh:.0f} KWh', COLORS['accent']),
    ('📅 Promedio/Día',  f'{avg_daily:.1f} KWh', COLORS['success']),
    ('🟢 Uptime',        f'{on_pct_global:.1f}%', COLORS['primary']),
    ('🚨 Anomalías',     f'{len(anom)} eventos', COLORS['warning']),
    ('🔺 Pico máximo',   f'{df["Consumption"].max():.2f} KWh', COLORS['warning']),
    ('📐 Skewness',      f'{df["Consumption"].skew():.3f}', COLORS['neutral']),
]
for i, (label, value, color) in enumerate(kpis):
    y_pos = 0.92 - i * 0.15
    ax3.text(0.05, y_pos, label, transform=ax3.transAxes,
             fontsize=11, color='#aaaacc', va='top')
    ax3.text(0.95, y_pos, value, transform=ax3.transAxes,
             fontsize=14, color=color, va='top', ha='right', fontweight='bold')
    ax3.axhline(y=y_pos - 0.04, color='#2a2a4a', linewidth=0.8,
                transform=ax3.get_xaxis_transform(), xmin=0, xmax=1)
ax3.set_title('KPIs del Análisis', fontsize=12, color=COLORS['accent'])

plt.savefig('10_resumen_ejecutivo.png', dpi=150, bbox_inches='tight',
            facecolor=COLORS['bg_dark'])
plt.show()
print('✅ Gráfica 10 generada — Análisis completo finalizado!')
print('\n📁 Archivos generados: 01_distribucion.png … 10_resumen_ejecutivo.png')